<a href="https://colab.research.google.com/github/ankit-rathi/Quantvesting_v2/blob/main/myStocks_Portfolio_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
'''
!pip install pandas==2.2.2 -qq
!pip install pyxirr -qq
!pip install ta -qq
!pip install yfinance==0.2.59 -qq
'''

'\n!pip install pandas==2.2.2 -qq\n!pip install pyxirr -qq\n!pip install ta -qq\n!pip install yfinance==0.2.59 -qq\n'

In [2]:
# import libraries

import numpy as np
import pandas as pd
import datetime
import warnings
warnings.filterwarnings('ignore')
import yfinance as yfin
import ta
import matplotlib.pyplot as plt
import requests

In [3]:
# notebook start time
import time
import datetime
import pytz

start_time = time.time()

# function to print date timestamp
def print_date_time():
  tz_NY = pytz.timezone('Asia/Kolkata')
  datetime_NY = datetime.datetime.now(tz_NY)
  print("Run date time (IST):", datetime_NY.strftime("%Y-%m-%d %H:%M:%S"))

In [4]:
# function to get booked and reserved amount
def get_amt():
  # fetch the JSON data from the URL
  url = "https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myPortfolioAmts.json"
  response = requests.get(url)
  pf_amts = response.json()  # parse the JSON data

  # extract values from the JSON
  py_booked_amt_dm = pf_amts["py_booked_amt_dm"]
  py_booked_amt_sv = pf_amts["py_booked_amt_sv"]
  cy_booked_amt_dm = pf_amts["cy_booked_amt_dm"]
  cy_booked_amt_sv = pf_amts["cy_booked_amt_sv"]
  reserve_amt_dm = pf_amts["reserve_amt_dm"]
  reserve_amt_sv = pf_amts["reserve_amt_sv"]

  # perform calculations
  py_booked_amt = py_booked_amt_dm + py_booked_amt_sv
  cy_booked_amt = cy_booked_amt_dm + cy_booked_amt_sv
  reserve_amt = reserve_amt_dm + reserve_amt_sv
  total_booked_amt = py_booked_amt + cy_booked_amt

  return total_booked_amt, reserve_amt, cy_booked_amt, py_booked_amt

gen_output = 0

In [5]:
# function to format the amount
def format_amt(number):
    abs_number = abs(number)

    if abs_number >= 1_00_00_000:  # Crores
        formatted_amt = f"{number / 1_00_00_000:.2f} C"
    elif abs_number >= 1_00_000:  # Lakhs
        formatted_amt = f"{number / 1_00_000:.2f} L"
    elif abs_number >= 1_000:  # Thousands
        formatted_amt = f"{number / 1_000:.2f} K"
    else:
        formatted_amt = f"{number:.2f}"

    return formatted_amt

# function to set start and end date
def get_start_end_date():
  start_date = (datetime.date.today() + datetime.timedelta(days=-365)).strftime('%Y-%m-%d')
  end_date = (datetime.date.today() + datetime.timedelta(days=1)).strftime('%Y-%m-%d')
  return start_date, end_date

# function to fetch my portfolio csv
def get_mypfs_df():
  mypfs_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myPortfolioStocks.csv')
  return mypfs_df

# function to fetch my prospects csv
def get_mypps_df():
  mypps_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myProspectsScrips.csv')
  return mypps_df

# function to fetch screener data
def get_myscreen_df():
  myscreen_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myScreenerDB.csv')
  return myscreen_df

# function to fetch momentum data
def get_myinvmt_df():
  myinvmt_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myInvestments.csv')
  return myinvmt_df

# function to get the stock ids
def get_stock_ids(df_pf):
  stock_n100 = df_pf['Symbol'].unique()

  exclude = ['CADILAHC','MMTC', 'MASFIN']

  stock_ids = df_pf[~df_pf['Symbol'].isin(exclude) ]['Symbol'].unique()

  #mypf = mypf[mypf.Forecast.notnull()]
  #stock_ids = mypf['Symbol'].unique()

  #stock_ids.sort()
  return stock_ids

# get features from screener data
def get_screener_features():
  myscreen_df = get_myscreen_df()
  cols = ['Symbol', 'EPS', 'MedPE', 'ROCE%', 'ROE%', 'CapType']
  return myscreen_df[cols]

# get relative strength
def get_relative_strength(stock_list):

    # Dictionary to store stock tickers and their corresponding percentage price change
    stock_changes = {}

    # Loop through each stock and fetch its price data
    for stock in stock_list:
        try:
            # Download the stock data for the given date range
            stock_data = yfin.Ticker(stock + '.NS').history(period='1mo', interval='1d')[map(str.title, ['open', 'close', 'low', 'high', 'volume'])]

            # Calculate the percentage change for the stock
            if len(stock_data) > 0:
                start_price = stock_data['Close'].iloc[0]
                end_price = stock_data['Close'].iloc[-1]
                percent_change = round(((end_price - start_price) / start_price) * 100, 2)
                stock_changes[stock] = percent_change
            else:
                stock_changes[stock] = np.nan  # If no data is available, set to NaN

        except Exception as e:
            print(f"Error fetching data for {stock}: {e}")
            stock_changes[stock] = np.nan

    # Create a DataFrame with stock tickers and their percentage changes
    df = pd.DataFrame(list(stock_changes.items()), columns=['Symbol', 'Percent_Change'])

    # Drop any stocks with missing data (NaN values)
    df = df.dropna()

    # Calculate the percentile rank based on percentage change
    df['RSP'] = round(df['Percent_Change'].rank(pct=True) * 100, 2)
    cols = ['Symbol', 'RSP']
    df = df[cols]
    # Sort by percentile rank (optional)
    df = df.sort_values(by='RSP', ascending=False).reset_index(drop=True)

    return df


In [6]:
# function to get stock technicals
def stock_prec_dev(stock_symbol):
    #stock_symbol = 'ULTRACEMCO.NS'
    short_window = 20
    mid_window = 50
    long_window = 200
    moving_avg = 'SMA'

    start = datetime.datetime(*map(int, start_date.split('-')))
    end = datetime.datetime(*map(int, end_date.split('-')))

    stock_df = yfin.Ticker(stock_symbol + '.NS').history(period='max', interval='1d')[map(str.title, ['open', 'close', 'low', 'high', 'volume'])]

    stock_df['Max'] = round(max(stock_df['Close']),2)
    stock_df = stock_df[(stock_df.index <= end_date) & (stock_df.index >= start_date)]
    stock_df['200_SMA'] = round(stock_df['Close'].rolling(window = 200, min_periods = 1).mean(),0)
    stock_df['Dev%_200'] = round((stock_df['Close'] - stock_df['200_SMA'])*100/stock_df['200_SMA'],2)
    stock_df.dropna(axis = 0, inplace = True) # remove any null rows

    stock_df['20_SMA'] = stock_df['Close'].rolling(window=20).mean()
    stock_df['50_SMA'] = stock_df['Close'].rolling(window=50).mean()
    stock_df['Symbol'] = stock_symbol

    stock_df['Close'] = round(stock_df['Close'],2)
    stock_df['Min'] = round(min(stock_df['Close']),2)
    stock_df['RSI_14'] = ta.momentum.RSIIndicator(close=stock_df['Close'], window=14).rsi()
    stock_df['RSI_14'] = round(stock_df['RSI_14'],0)
    stock_df['Prev_Close'] = stock_df['Close'].shift(1)
    stock_df.drop(['Open', 'Low', 'High', 'Volume'], axis=1, inplace=True)
    stock_df = stock_df.tail(1)

    max_SMA = max(stock_df['20_SMA'].item(), stock_df['50_SMA'].item(), stock_df['200_SMA'].item())
    min_SMA = min(stock_df['20_SMA'].item(), stock_df['50_SMA'].item(), stock_df['200_SMA'].item())
    ABS_Spread = max_SMA - min_SMA
    stock_df['Spread%'] = round((ABS_Spread / stock_df['200_SMA'].item()) * 100,2)

    return stock_df

# function to compute stock attributes
def get_common_features(stock_ids, df_mypf):

  df_prec_dev = pd.DataFrame()
  df_tmp = get_screener_features()
  df_rs = get_relative_strength(stock_ids)

  for stock_id in stock_ids:
      tmp = stock_prec_dev(stock_id)
      tmp = tmp.reset_index()
      df_prec_dev = pd.concat([df_prec_dev, tmp], ignore_index = True)
  df_prec_dev = pd.merge(df_prec_dev, df_mypf, on= 'Symbol')
  df_prec_dev = pd.merge(df_prec_dev, df_tmp, on= 'Symbol', how='left')
  df_prec_dev = pd.merge(df_prec_dev, df_rs, on= 'Symbol', how='left')
  #print(df_prec_dev.columns)
  df_prec_dev['Curr_PE'] = round(df_prec_dev['Close']/df_prec_dev['EPS'],1)
  df_prec_dev['Dev%_PE'] = round((df_prec_dev['Curr_PE'] - df_prec_dev['MedPE'])*100/df_prec_dev['MedPE'],2)
  df_prec_dev['Conviction'] = df_prec_dev['Conviction'] + '-' + df_prec_dev['CapType']
  return df_prec_dev

# function to arrange stock features
def arrange_features(df_stocks, common_cols, diff_cols):
  df_stocks_common = df_stocks[common_cols].drop_duplicates()
  df_stocks_diff = df_stocks[diff_cols]
  df_stocks_diff['Investment'] = df_stocks_diff['AvgCost'] * df_stocks_diff['Shares']
  df_stocks_diff = df_stocks_diff.groupby(['Symbol'])[['Shares', 'Investment']].aggregate(['sum']).reset_index()
  df_stocks_diff.columns = ['Symbol', 'Shares', 'Investment']
  df_stocks_diff['AvgCost'] = round(df_stocks_diff['Investment']/df_stocks_diff['Shares'],2)
  df_stocks = pd.merge(df_stocks_diff, df_stocks_common, on='Symbol')
  return df_stocks

# plot fact distribution across dimension
def plot_pie_chart(df, dimension, fact):
  # grouping the data by category and calculating the sum of fact for each type
  grouped_data = df.groupby(dimension)[fact].sum()

  # sorting the grouped data in descending order
  grouped_data = grouped_data.sort_values(ascending=False)

  # creating a pie chart
  grouped_data.plot.pie(autopct='%1.1f%%', startangle=90, figsize=(6, 6))

  # adding a title and displaying the plot
  plt.title(f'{dimension} {fact} Distribution')
  plt.ylabel('')  # To hide the y-label
  plt.show()

In [7]:
# function to get portfolio features

def get_portfolio_features(df_common_features):

  print_date_time()
  print('=================== PORTFOLIO QUALITY ====================')

  #df_common_features["Target"] = df_common_features["Target"].fillna(df_common_features["Max"])
  df_common_features['NTT'] = np.where(df_common_features['Strategy']== 'NTT', df_common_features["Target"], df_common_features['Max'])
  df_common_features['LTT'] = np.where(df_common_features['Strategy']== 'BTT', df_common_features["Target"], df_common_features['Max'])
  df_common_features['BOL'] = df_common_features['Min']

  tmp_df = df_common_features[df_common_features['Symbol'].isin(mypf_df[mypf_df['InPortfolio'] != 'NA'].Symbol.values)]
  print('qualified stocks: '+str(len(tmp_df['Symbol'].unique())))
  tmp_df1 = tmp_df[tmp_df['LatestQtr'] == 1]
  print('with latest results: '+str(len(tmp_df1['Symbol'].unique())))
  tmp_df1 = tmp_df1[tmp_df1['StarStock'] == 1]
  print('still star stocks: '+str(len(tmp_df1['Symbol'].unique())))
  tmp_df['Investment'] = tmp_df['AvgCost'] * tmp_df['Shares']
  tmp_df['Current'] = round(tmp_df['Close'] * tmp_df['Shares'],0)
  tmp_df['Previous'] = tmp_df['Prev_Close'] * tmp_df['Shares']
  tmp_df['EstimatedST'] = tmp_df['NTT'] * tmp_df['Shares']
  tmp_df['EstimatedLT'] = tmp_df['LTT'] * tmp_df['Shares']
  tmp_df['Current P/L'] = round((tmp_df['Current'] - tmp_df['Investment']),0)
  tmp_df['Today P/L%'] = round((tmp_df['Current'] - tmp_df['Previous'])*100/tmp_df['Previous'],2)
  tmp_df['Current P/L%'] = round((tmp_df['Current'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['EstimatedST P/L%'] = round((tmp_df['EstimatedST'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['EstimatedLT P/L%'] = round((tmp_df['EstimatedLT'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['NTT%'] = round((tmp_df['NTT'] - tmp_df['Close'])*100/tmp_df['Close'],2)
  tmp_df['LTT%'] = round((tmp_df['LTT'] - tmp_df['Close'])*100/tmp_df['Close'],2)
  tmp_df['Gained%'] = round((tmp_df['Close'] - tmp_df['BOL'])*100/tmp_df['BOL'],2)
  investment = round(sum(tmp_df['AvgCost']*tmp_df['Shares']),0)
  current = round(sum(tmp_df['Close']*tmp_df['Shares']),0)
  tmp_df['InitAlloc%'] = round(tmp_df['Investment']*100/investment,2)
  tmp_df['CurrAlloc%'] = round(tmp_df['Current']*100/current,2)
  tmp_df['FTT'] = tmp_df['LTT']
  tmp_df.loc[tmp_df['Strategy'] == 'NTT', 'FTT'] = tmp_df['NTT']
  tmp_df['FTT%'] = tmp_df['LTT%']
  tmp_df.loc[tmp_df['Strategy'] == 'NTT', 'FTT%'] = tmp_df['NTT%']
  tmp_df['FTT Amt'] = round(tmp_df['FTT%'] * tmp_df['Current']/100,0)
  tmp_df['OTT%'] = round((tmp_df['FTT'] - tmp_df['AvgCost'])*100/tmp_df['AvgCost'],2)
  tmp_df['RRR Ind'] = round(tmp_df['Current P/L']/tmp_df['FTT Amt'],2)
  tmp_df['Risk Ind'] = round(tmp_df['Current P/L%']*tmp_df['CurrAlloc%'],0)

  return tmp_df

# function to print portfolio features
def print_portfolio_stats(df_portfolio_features, myinvmt_df):
  from pyxirr import xirr

  total_booked_amt, reserve_amt, cy_booked_amt, py_booked_amt = get_amt()

  dates = myinvmt_df['Date'].values
  dates = np.append(dates, datetime.date.today().strftime('%d-%b-%y'))
  investment = myinvmt_df['Investment'].values
  dates= pd.to_datetime(dates)

  current = round(sum(df_portfolio_features['Close']*df_portfolio_features['Shares']),0) + reserve_amt
  investment_xirr = np.append(investment, current)
  cagr = round(xirr(pd.DataFrame({"dates": dates, "amounts": investment_xirr}))*100,2)

  investment = -sum(investment)
  invested = round(sum(df_portfolio_features['AvgCost']*df_portfolio_features['Shares']),0) + reserve_amt
  previous = round(sum(df_portfolio_features['Prev_Close']*df_portfolio_features['Shares']),0) + reserve_amt
  cy_invested = investment + py_booked_amt

  today_pnl_amount = current-previous
  today_pnl_percentage = round((current-previous)*100/previous,2)

  curr_pnl_amount = current-invested
  curr_pnl_percentage = round((curr_pnl_amount)*100/(cy_invested),2)

  cy_pnl_amount = cy_booked_amt + curr_pnl_amount
  cy_pnl_percentage = round((cy_pnl_amount)*100/cy_invested,2)

  overall_pnl_amount = total_booked_amt + curr_pnl_amount
  overall_pnl_percentage = round((overall_pnl_amount)*100/investment,2)

  estimate_st = round(sum(df_portfolio_features['FTT']*df_portfolio_features['Shares']),0)  + reserve_amt
  est_st_pnl_amount = estimate_st-current
  est_st_pnl_percentage = round((est_st_pnl_amount)*100/current,2)

  estimate_lt = round(sum(df_portfolio_features['LTT']*df_portfolio_features['Shares']),0)  + reserve_amt
  est_lt_pnl_amount = estimate_lt-current
  est_lt_pnl_percentage = round((est_lt_pnl_amount)*100/current,2)

  total_profit = round(sum(df_portfolio_features[df_portfolio_features['Current P/L%'] > 0]['Current']) - sum(df_portfolio_features[df_portfolio_features['Current P/L%'] > 0]['Investment']),0)
  total_loss = round(sum(df_portfolio_features[df_portfolio_features['Current P/L%'] < 0]['Current']) - sum(df_portfolio_features[df_portfolio_features['Current P/L%'] < 0]['Investment']),0)

  cy_booked_percentage = round((cy_booked_amt)*100/current,2)
  py_booked_percentage = round((py_booked_amt)*100/investment,2)
  total_booked_percentage = round((total_booked_amt)*100/investment,2)

  print('=================== PORTFOLIO SNAPSHOT ===================')
  print('Deployed Value: ', format_amt(investment))
  print('Current Value:  ', format_amt(current))
  print('CAGR/XIRR %:    ', '{}%'.format(cagr))
  print('Est FTT Value:  ', format_amt(estimate_st))
  print('Future Upside:  ', '{} ({}%)'.format(
      format_amt(est_st_pnl_amount),
      est_st_pnl_percentage
  ))

  print('=================== PERFORMANCE ==========================')
  print('Current PnL:             {} ({}%)'.format(
      format_amt(curr_pnl_amount), curr_pnl_percentage
  ))
  print('Total Booked PnL:        {} ({}%)'.format(
      format_amt(total_booked_amt), total_booked_percentage
  ))
  print('Booked + Current PnL:    {} ({}%)'.format(
      format_amt(overall_pnl_amount), overall_pnl_percentage
  ))
  print('CY Booked PnL:           {} ({}%)'.format(
      format_amt(cy_booked_amt), cy_booked_percentage
  ))
  print('Prev Year Booked PnL:    {} ({}%)'.format(
      format_amt(py_booked_amt), py_booked_percentage
  ))

  print('=================== TODAY / OPERATIONS ===================')
  print('Today PnL:               {} ({}%)'.format(
      format_amt(today_pnl_amount), today_pnl_percentage
  ))
  print('CY Investment:           ', format_amt(cy_invested))
  print('Reserve:                 ', format_amt(reserve_amt))

  print('=================== P&L DETAIL ============================')
  print('Total Profit:            ', format_amt(total_profit))
  print('Total Loss:              ', format_amt(total_loss))
  print('============================================================')

In [8]:
# get start and end date
start_date, end_date = get_start_end_date()

# get portfolio and prospects data
mypfs_df = get_mypfs_df()
mypps_df = get_mypps_df()
myinvmt_df = get_myinvmt_df()

# merge above datasets
mypf_df = pd.merge(mypfs_df, mypps_df, on="Symbol")

# seggregate dm and sv portfolio
dm_pf = mypf_df[mypf_df['InPortfolio'] == 'DM']
sv_pf = mypf_df[mypf_df['InPortfolio'] == 'SV']

# build portfolio stock dataframe
dm_stocks = get_stock_ids(dm_pf)
sv_stocks = get_stock_ids(sv_pf)
df_stocks = pd.concat([dm_pf,sv_pf], ignore_index = True)

# arrange common and diff stock features
common_cols = ['Symbol', 'Target', 'Criteria', 'Strategy', 'CumlRnk', 'LatestQtr', 'StarStock', 'Conviction', 'Category', 'InFolio', 'XIRR', 'MBQ']
diff_cols = ['Symbol', 'AvgCost', 'Shares']
df_stocks = arrange_features(df_stocks, common_cols, diff_cols)

# get common features
stock_ids = df_stocks['Symbol'].values
df_common_features = get_common_features(stock_ids, df_stocks)

df_common_features.reset_index(drop=True, inplace=True)
df_common_features.drop(['Date'], axis=1, inplace=True)
# get and print portfolio features
df_portfolio_features = get_portfolio_features(df_common_features)

print_portfolio_stats(df_portfolio_features, myinvmt_df)

df = df_portfolio_features
#plot_pie_chart(df, 'CapType', 'Current')
list_ox40n = list(df[df['MBQ'].str.contains('OX40N', na=False)]['Symbol'].values)

Run date time (IST): 2026-11-09 14:37:21
=================== PORTFOLIO QUALITY ====================
qualified stocks: 85
with latest results: 85
still star stocks: 45
=================== PORTFOLIO SNAPSHOT ===================
Deployed Value:  1.49 C
Current Value:   1.54 C
CAGR/XIRR %:     1.45%
Est FTT Value:   2.74 C
Future Upside:   1.20 C (77.68%)
=================== PERFORMANCE ==========================
Current PnL:             -42.91 L (-22.36%)
Total Booked PnL:        49.23 L (33.02%)
Booked + Current PnL:    6.31 L (4.24%)
CY Booked PnL:           6.36 L (4.12%)
Prev Year Booked PnL:    42.87 L (28.76%)
=================== TODAY / OPERATIONS ===================
Today PnL:               92.16 K (0.6%)
CY Investment:            1.92 C
Reserve:                  2.74 L
=================== P&L DETAIL ============================
Total Profit:             2.02 L
Total Loss:               -44.94 L


In [9]:
# top 5 near their targets
cols = ['Symbol', 'Today P/L%', 'Current P/L%', 'FTT%', 'OTT%', 'FTT Amt', 'Current P/L', 'Current', 'FTT', 'Dev%_PE', 'RSI_14','Conviction', 'Spread%',  'CumlRnk', 'RRR Ind', 'CurrAlloc%', 'Gained%', 'Criteria', 'Strategy', 'Category']
df_tmp = df_portfolio_features[~df_portfolio_features['Symbol'].isin(['ENRIN','BLUSPRING','DIGITIDE'])]
df_tmp = df_tmp.sort_values(by = 'FTT Amt', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
76,TTKPRESTIG,-1.24,-21.85,28.08,0.09,22115.0,-22020.0,78757.0,770.00,-4.89,70.0,M-SC,2.69,250.0,-1.00,0.52,41.75,OX40N,NTT,DURABLES
40,ICICIPRULI,2.63,-27.37,19.01,-13.56,27200.0,-53924.0,143081.0,522.35,-53.40,52.0,X-MC,9.52,78.0,-1.98,0.94,7.84,X40,ATH,INSURANCE
12,BOSCH-HCIL,1.35,-24.89,33.12,-0.01,31794.0,-31810.0,95995.0,2282.00,749.33,47.0,M-SC,19.81,231.0,-1.00,0.63,63.40,OX40N,BTT,AC
66,SIS,2.58,-10.06,34.83,21.26,34781.0,-11174.0,99858.0,528.00,92.07,50.0,H-SC,7.45,168.0,-0.32,0.66,48.47,OX40N,NTT,MISC
77,UNITDSPR,0.44,4.62,15.43,20.77,37798.0,10818.0,244962.0,1644.00,-10.79,60.0,X-MC,3.12,64.0,0.29,1.62,17.75,X40N,NTT,BREWERIES


In [10]:
# top 5 to accumulate
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC'])]
df_tmp = df_tmp[(df_tmp['CumlRnk'] < 40) & (df_tmp['FTT%'] > 20)].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
38,HYUNDAI,-0.43,9.70,31.21,43.94,86051.0,24385.0,275717.0,2782.73,13.61,65.0,X-LC,4.51,29.0,0.28,1.82,26.59,X40N,ATH,AUTO
52,MARUTI,0.77,-6.85,34.27,25.08,96127.0,-20618.0,280500.0,17120.06,-12.53,71.0,X-LC,12.95,33.0,-0.21,1.85,14.15,XR,ATH,AUTO
23,DMART,0.68,-15.30,56.36,32.43,169076.0,-54207.0,299993.0,5391.45,-29.56,54.0,X-LC,16.39,14.0,-0.32,1.98,10.03,X40N,ATH,FMCG
4,ASIANPAINT,-0.93,-8.57,35.69,24.06,108021.0,-28368.0,302664.0,3422.40,-19.82,65.0,X-LC,5.40,30.0,-0.26,2.00,20.21,X40,ATH,PAINTS
35,HDFCLIFE,0.04,-2.04,43.49,40.56,132559.0,-6355.0,304804.0,811.41,-28.65,65.0,X-LC,9.53,31.0,-0.05,2.01,10.11,X40,ATH,INSURANCE


In [11]:
# top 5 today
df_tmp = df_portfolio_features.sort_values(by = 'Today P/L%', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
62,REPCOHOME,20.69,-13.76,120.11,89.82,296763.0,-39429.0,247076.0,880.00,NaN,78.0,NaN,13.09,161.0,-0.13,1.63,31.66,XY24,NTT,FINANCE
25,FINCABLES,12.92,56.04,32.30,106.44,74414.0,82740.0,230384.0,1629.94,13.47,49.0,M-SC,24.55,220.0,1.11,1.52,74.56,OX40N,ATH,CABLES
27,GLAXO,6.48,7.93,25.73,35.69,55340.0,15796.0,215080.0,3380.16,-19.17,60.0,X-MC,8.97,56.0,0.29,1.42,27.83,X40,ATH,PHARMA
64,SFL,4.77,-45.30,131.52,26.63,188629.0,-118796.0,143422.0,1287.00,NaN,43.0,NaN,7.57,242.0,-0.63,0.95,19.70,XY24,NTT,MISC
3,ALKYLAMINE,3.71,-30.54,177.18,92.54,124262.0,-30830.0,70133.0,4520.85,NaN,42.0,NaN,7.57,142.0,-0.25,0.46,33.18,SR,ATH,CHEMICALS


In [12]:
# bottom 5 today
df_tmp = df_portfolio_features.sort_values(by = 'Today P/L%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
18,COFFEEDAY,-3.11,-48.43,162.30,35.27,95043.0,-54989.0,58560.0,80.00,NaN,53.0,NaN,1.62,266.0,-0.58,0.39,44.41,XR,NTT,HOTELS
45,INFY,-2.52,-16.69,71.24,42.66,237916.0,-66891.0,333964.0,1972.00,-39.69,58.0,X-LC,8.18,3.0,-0.28,2.20,19.56,X40,NTT,IT
33,HCLTECH,-2.42,-3.85,30.73,25.70,106659.0,-13889.0,347084.0,1837.07,-8.41,58.0,X-LC,7.83,8.0,-0.13,2.29,38.48,X40,ATH,IT
44,INDUSINDBK,-2.31,-34.35,102.43,32.89,51916.0,-26522.0,50684.0,1800.00,273.22,52.0,L-MC,6.38,258.0,-0.51,0.33,18.37,XR,NTT,BANKS
67,STARHEALTH,-2.09,7.00,37.63,47.26,113193.0,19682.0,300805.0,761.00,-5.75,54.0,H-SC,4.49,173.0,0.17,1.98,30.06,XY24,NTT,INSURANCE


In [13]:
# top 5 to exit based on CumlRnk
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['Current P/L%'] > -1) & (df_tmp['Current P/L%'] < 1)].sort_values(by = 'CumlRnk', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category


In [14]:
# OX40N stocks to exit
df_tmp = df_portfolio_features[(df_portfolio_features['Symbol'].isin(list_ox40n))].sort_values(by = 'Current P/L%', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,12.92,56.04,32.30,106.44,74414.0,82740.0,230384.0,1629.94,13.47,49.0,M-SC,24.55,220.0,1.11,1.52,74.56,OX40N,ATH,CABLES
66,SIS,2.58,-10.06,34.83,21.26,34781.0,-11174.0,99858.0,528.00,92.07,50.0,H-SC,7.45,168.0,-0.32,0.66,48.47,OX40N,NTT,MISC
76,TTKPRESTIG,-1.24,-21.85,28.08,0.09,22115.0,-22020.0,78757.0,770.00,-4.89,70.0,M-SC,2.69,250.0,-1.00,0.52,41.75,OX40N,NTT,DURABLES
12,BOSCH-HCIL,1.35,-24.89,33.12,-0.01,31794.0,-31810.0,95995.0,2282.00,749.33,47.0,M-SC,19.81,231.0,-1.00,0.63,63.40,OX40N,BTT,AC
49,KANSAINER,2.61,-32.87,69.04,13.47,124974.0,-88650.0,181017.0,340.00,-29.72,63.0,H-SC,6.72,162.0,-0.71,1.19,27.95,XY24,NTT,PAINTS


In [15]:
# top 5 to book for rotation
df_tmp = df_portfolio_features[(df_portfolio_features['Current P/L%'] > 20) ].sort_values(by = 'CumlRnk', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,12.92,56.04,32.3,106.44,74414.0,82740.0,230384.0,1629.94,13.47,49.0,M-SC,24.55,220.0,1.11,1.52,74.56,OX40N,ATH,CABLES


In [16]:
# top 5 to monitor for exit
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['Current P/L%'] > 0) ].sort_values(by = 'Dev%_PE', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,12.92,56.04,32.30,106.44,74414.0,82740.0,230384.0,1629.94,13.47,49.0,M-SC,24.55,220.0,1.11,1.52,74.56,OX40N,ATH,CABLES
67,STARHEALTH,-2.09,7.00,37.63,47.26,113193.0,19682.0,300805.0,761.00,-5.75,54.0,H-SC,4.49,173.0,0.17,1.98,30.06,XY24,NTT,INSURANCE


In [17]:
# top 5 to book for rotation from weak conviction
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[~df_tmp['Criteria'].isin(['XY25','XY24','X40', 'X40N'])].sort_values(by = 'RRR Ind', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,12.92,56.04,32.30,106.44,74414.0,82740.0,230384.0,1629.94,13.47,49.0,M-SC,24.55,220.0,1.11,1.52,74.56,OX40N,ATH,CABLES
84,WIPRO,-0.46,-9.43,113.11,93.01,256355.0,-23598.0,226642.0,420.00,-23.61,67.0,M-LC,7.63,98.0,-0.09,1.50,25.70,XR,NTT,IT
70,TANLA,-0.58,-26.79,193.93,115.17,384393.0,-72549.0,198212.0,1903.93,NaN,72.0,NaN,15.47,147.0,-0.19,1.31,75.22,AR,ATH,IT
3,ALKYLAMINE,3.71,-30.54,177.18,92.54,124262.0,-30830.0,70133.0,4520.85,NaN,42.0,NaN,7.57,142.0,-0.25,0.46,33.18,SR,ATH,CHEMICALS
42,INDIAMART,-0.17,-33.63,202.75,100.92,165957.0,-41483.0,81853.0,4675.60,NaN,43.0,NaN,16.80,140.0,-0.25,0.54,3.85,AR,ATH,MISC


In [18]:
# top 5 to accumulate based on lowest RSI_14
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'RSI_14', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
37,HONAUT,0.03,-21.98,93.40,50.88,140520.0,-42388.0,150450.0,58192.80,-39.00,36.0,X-SC,7.30,90.0,-0.30,0.99,14.63,X40N,ATH,ELECTRICAL
6,AWL,-0.20,-42.60,190.99,67.02,390582.0,-151805.0,204504.0,485.00,-57.79,38.0,X-SC,7.82,81.0,-0.39,1.35,1.72,XY24,NTT,FMCG
28,GODREJCP,-0.56,-23.02,71.60,32.10,137543.0,-57438.0,192099.0,1452.16,-24.40,49.0,X-MC,16.38,32.0,-0.42,1.27,4.08,X40N,ATH,FMCG
11,BERGEPAINT,-1.49,-13.16,44.31,25.32,99175.0,-33929.0,223820.0,680.00,-24.45,49.0,X-MC,2.17,68.0,-0.34,1.48,17.37,XY24,NTT,PAINTS
20,DABUR,-0.69,-26.83,96.39,43.70,174958.0,-66557.0,181511.0,735.00,-36.80,50.0,X-MC,12.73,70.0,-0.38,1.20,5.57,XY24,BTT,FMCG


In [19]:
# top 5 to accumulate based on Spread and CumlRnk
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['CumlRnk'] < 100)].sort_values(by = 'Spread%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
17,CIPLA,-0.57,5.46,17.47,23.89,55394.0,16423.0,317078.0,1648.14,19.62,63.0,X-LC,0.82,18.0,0.30,2.09,18.77,X40N,ATH,PHARMA
1,ABBOTINDIA,1.20,-10.56,29.00,15.37,46388.0,-18895.0,159960.0,34390.26,-27.16,56.0,X-MC,1.03,58.0,-0.41,1.06,8.36,X40,ATH,PHARMA
11,BERGEPAINT,-1.49,-13.16,44.31,25.32,99175.0,-33929.0,223820.0,680.00,-24.45,49.0,X-MC,2.17,68.0,-0.34,1.48,17.37,XY24,NTT,PAINTS
55,PAGEIND,1.00,-5.35,40.36,32.85,102527.0,-14349.0,254030.0,50936.06,-29.61,56.0,X-MC,2.17,55.0,-0.14,1.68,21.30,X40,ATH,APPARELS
77,UNITDSPR,0.44,4.62,15.43,20.77,37798.0,10818.0,244962.0,1644.00,-10.79,60.0,X-MC,3.12,64.0,0.29,1.62,17.75,X40N,NTT,BREWERIES


In [20]:
# top 5 to accumulate based on lowest Gained%
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'Gained%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
6,AWL,-0.20,-42.60,190.99,67.02,390582.0,-151805.0,204504.0,485.00,-57.79,38.0,X-SC,7.82,81.0,-0.39,1.35,1.72,XY24,NTT,FMCG
28,GODREJCP,-0.56,-23.02,71.60,32.10,137543.0,-57438.0,192099.0,1452.16,-24.40,49.0,X-MC,16.38,32.0,-0.42,1.27,4.08,X40N,ATH,FMCG
20,DABUR,-0.69,-26.83,96.39,43.70,174958.0,-66557.0,181511.0,735.00,-36.80,50.0,X-MC,12.73,70.0,-0.38,1.20,5.57,XY24,BTT,FMCG
26,GILLETTE,-0.51,-12.93,47.80,28.69,105664.0,-32841.0,221055.0,10891.01,-37.24,56.0,X-SC,6.16,83.0,-0.31,1.46,6.81,X40,ATH,FMCG
19,COLPAL,0.45,-35.96,115.11,37.76,194145.0,-94705.0,168660.0,3628.04,-22.52,50.0,X-MC,14.55,62.0,-0.49,1.11,7.26,XY25,ATH,FMCG


In [21]:
# top 5 to accumulate based on lowest CurrAlloc%
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
56,PGHH,2.15,-46.56,145.15,31.02,165634.0,-99408.0,114112.0,17484.40,-60.69,55.0,X-SC,23.80,57.0,-0.60,0.75,7.43,X40,ATH,FMCG
40,ICICIPRULI,2.63,-27.37,19.01,-13.56,27200.0,-53924.0,143081.0,522.35,-53.40,52.0,X-MC,9.52,78.0,-1.98,0.94,7.84,X40,ATH,INSURANCE
2,ACC,-0.24,-44.81,216.74,74.81,320741.0,-120144.0,147984.0,3906.00,-45.40,61.0,X-SC,14.94,85.0,-0.37,0.98,9.63,XY24,BTT,CEMENT
37,HONAUT,0.03,-21.98,93.40,50.88,140520.0,-42388.0,150450.0,58192.80,-39.00,36.0,X-SC,7.30,90.0,-0.30,0.99,14.63,X40N,ATH,ELECTRICAL
1,ABBOTINDIA,1.20,-10.56,29.00,15.37,46388.0,-18895.0,159960.0,34390.26,-27.16,56.0,X-MC,1.03,58.0,-0.41,1.06,8.36,X40,ATH,PHARMA


In [22]:
# top 5 to accumulate based on CumlRnk
df_tmp = df_portfolio_features.sort_values(by = 'CumlRnk', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
47,ITC,-1.40,-20.50,61.89,28.70,215305.0,-89712.0,347883.0,452.00,-28.31,63.0,X-LC,7.42,1.0,-0.42,2.30,9.49,X40,NTT,FMCG
72,TCS,-1.51,-25.88,71.02,26.75,256906.0,-126337.0,361738.0,4208.34,-39.95,58.0,X-LC,6.92,2.0,-0.49,2.39,25.46,X40,ATH,IT
45,INFY,-2.52,-16.69,71.24,42.66,237916.0,-66891.0,333964.0,1972.00,-39.69,58.0,X-LC,8.18,3.0,-0.28,2.20,19.56,X40,NTT,IT
74,TMPV,0.94,-28.25,103.56,46.05,332410.0,-126389.0,320983.0,600.00,-85.99,62.0,X-LC,16.75,4.0,-0.38,2.12,12.84,XY24,NTT,AUTO
33,HCLTECH,-2.42,-3.85,30.73,25.70,106659.0,-13889.0,347084.0,1837.07,-8.41,58.0,X-LC,7.83,8.0,-0.13,2.29,38.48,X40,ATH,IT


In [23]:
# top 5 for average up
df_tmp = df_portfolio_features[(df_portfolio_features['Dev%_200'] > 5)].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
76,TTKPRESTIG,-1.24,-21.85,28.08,0.09,22115.0,-22020.0,78757.0,770.00,-4.89,70.0,M-SC,2.69,250.0,-1.00,0.52,41.75,OX40N,NTT,DURABLES
12,BOSCH-HCIL,1.35,-24.89,33.12,-0.01,31794.0,-31810.0,95995.0,2282.00,749.33,47.0,M-SC,19.81,231.0,-1.00,0.63,63.40,OX40N,BTT,AC
80,VALIANTORG,-1.04,-27.94,141.67,74.15,167022.0,-45710.0,117895.0,838.00,NaN,35.0,NaN,42.00,145.0,-0.27,0.78,75.45,XR,NTT,CHEMICALS
70,TANLA,-0.58,-26.79,193.93,115.17,384393.0,-72549.0,198212.0,1903.93,NaN,72.0,NaN,15.47,147.0,-0.19,1.31,75.22,AR,ATH,IT
27,GLAXO,6.48,7.93,25.73,35.69,55340.0,15796.0,215080.0,3380.16,-19.17,60.0,X-MC,8.97,56.0,0.29,1.42,27.83,X40,ATH,PHARMA


In [24]:
# top 5 RSP
df_tmp = df_portfolio_features.sort_values(by = 'RSP', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
70,TANLA,-0.58,-26.79,193.93,115.17,384393.0,-72549.0,198212.0,1903.93,NaN,72.0,NaN,15.47,147.0,-0.19,1.31,75.22,AR,ATH,IT
71,TATAELXSI,0.82,-43.99,135.70,32.03,100210.0,-57988.0,73847.0,9161.00,-36.36,64.0,H-SC,13.26,158.0,-0.58,0.49,29.73,OX40N,NTT,IT
84,WIPRO,-0.46,-9.43,113.11,93.01,256355.0,-23598.0,226642.0,420.00,-23.61,67.0,M-LC,7.63,98.0,-0.09,1.50,25.70,XR,NTT,IT
62,REPCOHOME,20.69,-13.76,120.11,89.82,296763.0,-39429.0,247076.0,880.00,NaN,78.0,NaN,13.09,161.0,-0.13,1.63,31.66,XY24,NTT,FINANCE
13,BSOFT,-1.35,-39.66,143.96,47.20,122170.0,-55789.0,84864.0,811.90,NaN,61.0,NaN,10.81,172.0,-0.46,0.56,25.24,XR,ATH,IT


In [25]:
# Top N allocation
df_tmp = df_portfolio_features.sort_values(by = 'CurrAlloc%', ascending=False)
top_n_values = [10, 25, 50]

sum_df = pd.DataFrame({
    'Top_N': top_n_values,
    'Sum_Alloc%': [df_tmp['CurrAlloc%'].head(n).sum() for n in top_n_values]
})

sum_df

,Top_N,Sum_Alloc%
0,10,22.40
1,25,50.32
2,50,81.19


In [26]:
# market-cap-wise allocation
df_tmp = df_portfolio_features[cols]
df_tmp.groupby(df_tmp['Conviction'].str[-2:])['CurrAlloc%'].sum().sort_values(ascending=False)

Conviction
LC    41.75
MC    20.89
SC    20.48
Name: CurrAlloc%, dtype: float64

In [27]:
# criteria-wise allocation
df_portfolio_features.groupby('Criteria')['CurrAlloc%'].sum().sort_values(ascending=False)

Criteria
X40      27.99
XY24     22.91
X40N     18.91
XR        9.80
XY25      9.60
OX40N     6.81
AR        3.22
SR        0.76
Name: CurrAlloc%, dtype: float64

In [28]:
# conviction-wise allocation
df_portfolio_features.groupby('Conviction')['CurrAlloc%'].sum().sort_values(ascending=False)

Conviction
X-LC    40.25
X-MC    18.47
H-SC     8.24
X-SC     7.08
M-SC     5.16
M-LC     1.50
M-MC     1.10
H-MC     0.99
L-MC     0.33
Name: CurrAlloc%, dtype: float64

In [29]:
# sector-wise stats
df_tmp = df_portfolio_features.groupby('Category')[['CurrAlloc%', 'Current', 'Current P/L', 'FTT Amt']].sum().sort_values(by=['Current', 'Current P/L'], ascending=False)
df_tmp['Current P/L%'] = round(df_tmp['Current P/L'] * 100 / df_tmp['Current'], 2)
df_tmp['FTT%'] = round(df_tmp['FTT Amt'] * 100 / df_tmp['Current'], 2)
cols = ['CurrAlloc%', 'Current P/L%', 'FTT%']
df_tmp[cols].sort_values(by=['CurrAlloc%'], ascending=False)

,CurrAlloc%,Current P/L%,FTT%
Category,,,
FMCG,17.87,-26.67,72.14
IT,12.94,-30.86,99.05
FINANCE,11.02,-20.70,72.26
AUTO,8.73,-13.29,53.70
INSURANCE,6.08,-7.14,36.28
MISC,5.55,-46.85,105.88
PAINTS,5.42,-25.80,47.88
BANKS,5.16,-14.50,71.92
PHARMA,4.57,1.93,22.70


In [30]:
# money to be made criteria-wise
df_portfolio_features.groupby('Criteria')['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,sum,count
Criteria,,
XY24,3963174.0,19
X40,2150036.0,17
XR,1568299.0,13
X40N,1470599.0,12
AR,1019690.0,6
OX40N,758917.0,10
XY25,757820.0,6
SR,298055.0,2


In [31]:
# money to be made conviction-wise
df_portfolio_features.groupby('Conviction')['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,sum,count
Conviction,,
X-LC,2840919.0,19
X-MC,1608728.0,14
X-SC,1201068.0,6
H-SC,900058.0,8
M-SC,379974.0,6
H-MC,263946.0,1
M-LC,256355.0,1
M-MC,78486.0,1
L-MC,51916.0,1


In [32]:
# money to be made criteria and conviction-wise
df_portfolio_features.groupby(['Conviction', 'Criteria'])['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

sum  count
Conviction Criteria                  
X-LC       X40       1274840.0      8
X-SC       XY24       711323.0      2
X-LC       XY24       555778.0      2
H-SC       XY24       552276.0      3
X-LC       X40N       545230.0      5
X-MC       X40N       541141.0      4
           X40        483064.0      6
X-LC       XY25       368944.0      3
X-MC       XY25       310390.0      2
           XY24       274133.0      2
X-SC       X40        271298.0      2
H-MC       XY24       263946.0      1
M-LC       XR         256355.0      1
H-SC       OX40N      236663.0      3
X-SC       X40N       218447.0      2
M-SC       OX40N      189279.0      4
           XY24       130379.0      1
H-SC       XR         111119.0      2
X-LC       XR          96127.0      1
M-MC       XY25        78486.0      1
M-SC       XR          60316.0      1
L-MC       XR          51916.0      1

In [33]:
# notebook execution time

end_time = time.time()
execution_time = round(end_time - start_time, 0)
print(f"Notebook execution time: {execution_time} seconds")

Notebook execution time: 26.0 seconds
